# 01 — Data Collection

Pulls Fed-rate-decision event markets: metadata, bid/ask price history, and
correlated macro features (yields, CPI/jobs surprises, VIX-style vol).

Two modes:
- `use_live=True` — pulls real markets from Polymarket's public Gamma/CLOB APIs
- `use_live=False` (default here) — generates structurally realistic synthetic
  data, so this notebook runs end-to-end with no network access. Flip the flag
  and re-run when you have internet access to work with real markets.


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from data_loader import load_data

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [ ]:
markets_df, price_df, macro_df = load_data(use_live=False)

print(f"markets_df: {markets_df.shape}")
print(f"price_df:   {price_df.shape}")
print(f"macro_df:   {macro_df.shape}")

## Markets

In [ ]:
markets_df.head(10)

In [ ]:
markets_df["market_id"].nunique(), markets_df["category"].value_counts()

## Raw price history (bid/ask/volume)

In [ ]:
price_df.head(10)

In [ ]:
price_df.describe()

## Macro features

In [ ]:
macro_df.head(10)

## Persist to `data/raw/` for downstream notebooks

In [ ]:
import os
os.makedirs("../data/raw", exist_ok=True)

markets_df.to_csv("../data/raw/markets.csv", index=False)
price_df.to_csv("../data/raw/price_history.csv", index=False)
macro_df.to_csv("../data/raw/macro_features.csv", index=False)

print("Saved markets.csv, price_history.csv, macro_features.csv to data/raw/")

### Notes on swapping in live Kalshi/Polymarket data

- Polymarket: `PolymarketClient.search_markets()` + `get_price_history()` in
  `src/data_loader.py` hit the real Gamma/CLOB REST APIs — no auth needed for
  read-only market data.
- Kalshi: the same shape (`market_id`, `event_title`, `outcome_name`,
  `expiration_date`, `bid`, `ask`, `volume`) can be produced from Kalshi's
  public REST API (`/trade-api/v2/markets`, `/trade-api/v2/markets/{ticker}/orderbook`).
  Swap the client, keep everything downstream (`probability_cleaning.py` onward)
  unchanged — that's the point of standardizing on this schema.
